# 04 - Testing & Statistical Evaluation (Unified Harness)

Every model's test-set predictions are generated with the SAME
`evaluate_walk_forward` function used during CV tuning (Notebook 02) --
single fit on the tuning pool, then a forecast generated at every point across
the held-out test block using only data strictly before that point. This
guarantees every model is scored on **identical, time-aligned target windows**,
which is a hard requirement for the Diebold-Mariano test's paired-error
assumption to be valid.

Runs Diebold-Mariano significance tests (ELinear vs. every benchmark) with
Holm-Bonferroni correction for multiple comparisons, and produces the raw
scored results consumed by Notebook 05 for publication-ready tables/figures.

## Colab setup
Run this once per session. It mounts Google Drive (where the whole `potato_pipeline/` project folder should be uploaded), `cd`s into `notebooks/` so every relative path (`../src`, `../data`, `../results`) keeps working unchanged, and installs the extra packages this notebook needs (Colab's base image doesn't include them).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Adjust this if you uploaded the project folder somewhere else in Drive
PROJECT_ROOT = "/content/drive/MyDrive/potato_pipeline"
os.chdir(os.path.join(PROJECT_ROOT, "notebooks"))
print("Working directory:", os.getcwd())

In [ ]:
!pip install -q pmdarima neuralforecast

In [ ]:
import sys, os, json
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings("ignore")

import cv_utils
import elinear_model
import nf_wrappers
from metrics import summarize_metrics, diebold_mariano, holm_bonferroni, mse

RESULTS_DIR = "../results"
PRED_DIR = "../results/predictions"
NF_CKPT_DIR = "../results/nf_checkpoints"
HORIZONS = [1, 7, 14, 30]
SEEDS = [0, 1, 2, 3, 4]
DL_MODEL_KEYS = ["ELinear", "LSTM", "TCN", "NBEATS", "DLinear", "NLinear", "PatchTST", "iTransformer"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

df = pd.read_csv("../data/agra_potato_clean.csv", index_col=0, parse_dates=True)
series = df["price"].values
dates = df.index
n = len(series)

pool_idx, test_idx = cv_utils.chronological_holdout(n, test_frac=0.15)
series_pool_len = len(pool_idx)
print(f"Test block: {dates[test_idx[0]].date()} to {dates[test_idx[-1]].date()} ({len(test_idx)} days)")

## 1. Score all DL models via the shared walk-forward evaluator

`evaluate_walk_forward(model, predict_fn, series, seq_len, pred_len, train_end, val_end)` is called with `train_end = series_pool_len` (end of training data) and `val_end = n` (end of the full series), so every model forecasts across exactly the same held-out test range.

In [ ]:
all_predictions = {}  # all_predictions[model_name][horizon] = list of (y_pred_array, y_true_array) per seed

# Same normalization stats as Notebook 03 -- deterministic given the same
# series_pool, so recomputing here (rather than persisting across notebooks)
# reproduces the exact stats the checkpoints were fit on.
pool_mean, pool_std = cv_utils.zscore_stats(series[:series_pool_len])
print(f"Normalization stats (from tuning pool): mean={pool_mean:.2f}, std={pool_std:.2f}")

def load_elinear(h, seed):
    ckpt = torch.load(f"{PRED_DIR}/elinear_h{h}_seed{seed}.pt", map_location="cpu")
    model = elinear_model.ELinear(seq_len=ckpt["seq_len"], pred_len=h, n_learners=ckpt["n_learners"])
    model.load_state_dict(ckpt["state_dict"])
    return model, ckpt["seq_len"]

from neuralforecast import NeuralForecast
def load_nf(model_key, h, seed):
    nf = NeuralForecast.load(f"{NF_CKPT_DIR}/{model_key}_h{h}_seed{seed}")
    return nf

for model_key in DL_MODEL_KEYS:
    with open(f"{RESULTS_DIR}/best_hp_{model_key.lower()}.json") as f:
        hp = json.load(f)
    all_predictions[model_key] = {}

    for h_str, cfg in hp.items():
        h = int(h_str)
        seq_len = cfg["params"]["seq_len"]
        seed_results = []

        for seed in SEEDS:
            if model_key == "ELinear":
                model, sl = load_elinear(h, seed)
                predict_fn = elinear_model.elinear_predict_fn
            else:
                model = load_nf(model_key, h, seed)
                predict_fn = nf_wrappers.nf_predict_fn
                sl = seq_len

            preds, trues = cv_utils.evaluate_walk_forward(
                model, predict_fn, series, sl, h, train_end=series_pool_len, val_end=n,
                norm_stats=(pool_mean, pool_std),
            )
            if preds is not None:
                seed_results.append((preds, trues))

        all_predictions[model_key][h] = seed_results
        print(f"[{model_key}] horizon={h}: scored {len(seed_results)} seed(s), "
              f"{seed_results[0][0].shape[0] if seed_results else 0} test windows")

## 2. Naive & Seasonal-Naive (single deterministic prediction each)

In [ ]:
def naive_predict_fn(model, context, seq_len, pred_len):
    return np.full(pred_len, context[-1])

def seasonal_naive_predict_fn(model, context, seq_len, pred_len, period=365):
    n_ctx = len(context)
    if n_ctx < period:
        return np.full(pred_len, context[-1])
    start = n_ctx - period
    end = start + pred_len
    if end <= n_ctx:
        return context[start:end]
    avail = context[start:n_ctx]
    pad = np.full(pred_len - len(avail), context[-1])
    return np.concatenate([avail, pad])

for name, pfn in [("Naive", naive_predict_fn), ("SeasonalNaive", seasonal_naive_predict_fn)]:
    all_predictions[name] = {}
    for h in HORIZONS:
        preds, trues = cv_utils.evaluate_walk_forward(
            None, pfn, series, seq_len=1, pred_len=h, train_end=series_pool_len, val_end=n
        )
        all_predictions[name][h] = [(preds, trues)] if preds is not None else []
    print(f"Scored {name}.")

## 3. ARIMA

Refit once per horizon on the full tuning pool (deterministic, cheap), then
scored via the same walk-forward evaluator for consistency with every other
model's test-window alignment.

In [ ]:
import pmdarima as pm
from pmdarima.preprocessing import FourierFeaturizer

# NOTE -- bug fix: pmdarima's .predict() returns a forecast anchored to
# wherever the model was last fit/updated; it does NOT use the `context`
# argument a plain predict_fn receives on its own. Calling .predict() inside
# a walk-forward loop without ever telling the model about newly elapsed
# observations silently returns the SAME stale forecast at every step
# (verified: identical output for context lengths 250 and 290 on synthetic
# data). The fix is to call `model.update(new_y, X=new_exog)` with only the
# newly observed points since the last call, advancing the model's internal
# state before each prediction -- this is pmdarima's supported mechanism for
# genuine walk-forward evaluation without a full refit at every step.
#
# ARIMA is also fit and updated on z-score normalized data (see Notebook 02),
# for the same input-treatment consistency applied to every other model.

class ArimaWalkForwardState:
    """Tracks the ARIMA model's fitted state and how much of the series it has
    already seen, so `update()` is called with only the newly elapsed points."""
    def __init__(self, model, fourier, mean, std, last_seen_end):
        self.model = model
        self.fourier = fourier
        self.mean = mean
        self.std = std
        self.last_seen_end = last_seen_end  # index into the RAW series

def make_arima_predict_fn(state, raw_series):
    def predict_fn(_model_unused, context, seq_len, pred_len):
        current_end = len(context)  # context = raw_series[:t], so this is t
        if current_end > state.last_seen_end:
            full_ctx_n = cv_utils.zscore_transform(raw_series[:current_end], state.mean, state.std)
            _, exog_full = state.fourier.fit_transform(pd.Series(full_ctx_n))
            new_y = np.asarray(full_ctx_n)[state.last_seen_end:current_end]
            new_exog = np.asarray(exog_full)[state.last_seen_end:current_end]
            try:
                state.model.update(new_y, X=new_exog)
            except Exception:
                pass  # keep previous state rather than crash the whole walk-forward loop
            state.last_seen_end = current_end
        try:
            ctx_n = cv_utils.zscore_transform(context, state.mean, state.std)
            _, exog_future = state.fourier.transform(pd.Series(ctx_n), n_periods=pred_len)
            fc_n = state.model.predict(n_periods=pred_len, exogenous=exog_future)
            return cv_utils.zscore_inverse(np.asarray(fc_n)[:pred_len], state.mean, state.std)
        except Exception:
            return np.full(pred_len, context[-1])  # fallback to naive on numerical failure
    return predict_fn

all_predictions["ARIMA"] = {}
for h in HORIZONS:
    mean, std = cv_utils.zscore_stats(series[:series_pool_len])
    train_n = cv_utils.zscore_transform(series[:series_pool_len], mean, std)
    fourier = FourierFeaturizer(m=365, k=3)
    y_prime, exog = fourier.fit_transform(pd.Series(train_n))
    fitted_model = pm.auto_arima(y_prime, exogenous=exog, seasonal=True, m=7,
                                  stepwise=True, information_criterion="aicc",
                                  suppress_warnings=True, error_action="ignore")

    state = ArimaWalkForwardState(fitted_model, fourier, mean, std, last_seen_end=series_pool_len)
    predict_fn = make_arima_predict_fn(state, series)
    preds, trues = cv_utils.evaluate_walk_forward(
        None, predict_fn, series, seq_len=max(h * 2, 2), pred_len=h,
        train_end=series_pool_len, val_end=n,
    )
    all_predictions["ARIMA"][h] = [(preds, trues)] if preds is not None else []
    print(f"[ARIMA] horizon={h}: scored {len(all_predictions['ARIMA'][h])} seed-equivalent run(s)")

## 4. Aggregate metrics table (mean +/- std across seeds)

In [ ]:
rows = []
for model_name, by_h in all_predictions.items():
    for h, seed_results in by_h.items():
        if not seed_results:
            continue
        seed_metrics = []
        for y_pred, y_true in seed_results:
            m = summarize_metrics(y_true.flatten(), y_pred.flatten())
            seed_metrics.append(m)
        metric_names = seed_metrics[0].keys()
        agg = {"model": model_name, "horizon": h}
        for mname in metric_names:
            vals = [sm[mname] for sm in seed_metrics]
            agg[f"{mname}_mean"] = np.mean(vals)
            agg[f"{mname}_std"] = np.std(vals)
        rows.append(agg)

metrics_table = pd.DataFrame(rows).sort_values(["horizon", "MSE_mean"])
metrics_table.to_csv(f"{RESULTS_DIR}/test_metrics_full.csv", index=False)
metrics_table

## 4b. Export prediction-vs-actual series for plotting (Notebook 05)

For each horizon, save the h-day-ahead forecast aligned to the calendar date it targets, for the 9 models on a comparable error scale (Naive, LSTM, TCN, N-BEATS, DLinear, NLinear, PatchTST, iTransformer, ELinear -- ARIMA and Seasonal-Naive excluded here for the same reason as Figure 2: their error is roughly 200x larger and would flatten every other line to the baseline on a shared axis).

**Seed choice:** rather than averaging predictions across the 5 seeds (which can smooth over real model behavior), the single seed whose overall MSE is closest to the 5-seed mean is used -- an actual model run, not a synthetic average.

In [ ]:
PLOT_MODELS = ["Naive", "LSTM", "TCN", "NBEATS", "DLinear", "NLinear", "PatchTST", "iTransformer", "ELinear"]

def representative_seed_index(seed_results):
    """Return the index of the seed whose MSE is closest to the mean MSE
    across all seeds for this model/horizon (an actual run, not an average)."""
    mses = [mse(t.flatten(), p.flatten()) for p, t in seed_results]
    mean_mse = np.mean(mses)
    return int(np.argmin([abs(m - mean_mse) for m in mses]))

for h in HORIZONS:
    # Target date for window i is the LAST day of its pred_len-length forecast:
    # window i covers series[series_pool_len + i : series_pool_len + i + h],
    # so its h-day-ahead target date is dates[series_pool_len + i + h - 1].
    n_windows = None
    series_dict = {}
    actual_series = None

    for model_name in PLOT_MODELS:
        seed_results = all_predictions.get(model_name, {}).get(h, [])
        if not seed_results:
            print(f"No results for {model_name} at h={h} -- skipping")
            continue
        idx = representative_seed_index(seed_results)
        preds, trues = seed_results[idx]
        h_step_pred = preds[:, -1]
        h_step_true = trues[:, -1]

        if n_windows is None:
            n_windows = len(h_step_pred)
            actual_series = h_step_true
        elif len(h_step_pred) != n_windows:
            print(f"Skipping {model_name} h={h}: window count mismatch")
            continue

        series_dict[model_name] = h_step_pred

    if n_windows is None:
        continue

    target_dates = dates[series_pool_len + h - 1: series_pool_len + h - 1 + n_windows]
    out_df = pd.DataFrame({"date": target_dates, "actual": actual_series})
    for model_name, vals in series_dict.items():
        out_df[model_name] = vals

    out_path = f"{RESULTS_DIR}/pred_vs_actual_h{h}.csv"
    out_df.to_csv(out_path, index=False)
    print(f"Saved {out_path} ({len(out_df)} rows, {len(series_dict)} models)")

## 5. Diebold-Mariano significance tests -- full pairwise matrix

Every pair of models is compared at every horizon (not just ELinear vs. the rest), since Table 3 showed no single model wins every horizon -- a reader will reasonably want to know, e.g., whether DLinear is significantly better than PatchTST at h = 14 d, not only how each compares to ELinear. Holm-Bonferroni correction is applied across the full family of pairwise tests within each horizon.

In [ ]:
import itertools

dm_rows = []
for h in HORIZONS:
    # Every model that has a prediction at this horizon, in a fixed order
    models_at_h = [m for m, by_h in all_predictions.items() if h in by_h and by_h[h]]
    if len(models_at_h) < 2:
        continue

    # Seed-averaged predictions and shared true values per model (all models
    # were scored on IDENTICAL windows -- verified elsewhere in this notebook
    # -- so y_true is interchangeable across models at a given horizon).
    preds_by_model = {}
    y_true_ref = None
    for m in models_at_h:
        p = np.mean([pr for pr, _ in all_predictions[m][h]], axis=0)
        t = all_predictions[m][h][0][1]
        if y_true_ref is None:
            y_true_ref = t
        elif p.shape != y_true_ref.shape:
            print(f"Skipping {m} h={h}: shape mismatch ({p.shape} vs {y_true_ref.shape})")
            continue
        preds_by_model[m] = p

    models_at_h = list(preds_by_model.keys())
    errors_by_model = {m: (y_true_ref - preds_by_model[m]).flatten() for m in models_at_h}

    # Full pairwise comparison -- every (model_a, model_b) combination, not just
    # ELinear vs. the rest. This matters because no single model dominated
    # every horizon (Table 3): reporting only "ELinear vs. benchmark" pairs
    # would hide, e.g., whether DLinear is significantly better than PatchTST.
    pair_list = list(itertools.combinations(models_at_h, 2))
    pvals, stats_ = [], []
    for model_a, model_b in pair_list:
        dm_stat, p_val = diebold_mariano(errors_by_model[model_a], errors_by_model[model_b], h=h, power=2)
        stats_.append(dm_stat); pvals.append(p_val)

    # Holm-Bonferroni correction across the FULL family of pairwise tests
    # within this horizon (all C(n,2) pairs), not just the smaller
    # ELinear-vs-9 family used previously -- the more pairs tested, the more
    # correction is needed to control the family-wise error rate.
    reject, adj_p = holm_bonferroni(pvals, alpha=0.05)
    for (model_a, model_b), stat, raw_p, adj, rej in zip(pair_list, stats_, pvals, adj_p, reject):
        better = None
        if rej:
            better = model_a if stat < 0 else model_b  # negative DM stat -> model_a has lower squared error
        dm_rows.append({
            "horizon": h, "model_a": model_a, "model_b": model_b,
            "dm_stat": stat, "p_value_raw": raw_p, "p_value_holm": adj,
            "significant": rej, "better_model": better,
        })

dm_pairwise = pd.DataFrame(dm_rows)
dm_pairwise.to_csv(f"{RESULTS_DIR}/dm_pairwise_full.csv", index=False)
print(f"Computed {len(dm_pairwise)} pairwise DM tests across {len(HORIZONS)} horizons.")

# Convenience subset: ELinear vs. every other model (kept for backward
# compatibility with any existing manuscript text referencing this framing;
# the full matrix above is the primary output going forward).
dm_table = dm_pairwise[(dm_pairwise.model_a == "ELinear") | (dm_pairwise.model_b == "ELinear")].copy()
dm_table["benchmark"] = dm_table.apply(lambda r: r.model_b if r.model_a == "ELinear" else r.model_a, axis=1)
dm_table["elinear_significantly_better"] = dm_table["significant"] & (dm_table["better_model"] == "ELinear")
dm_table = dm_table[["horizon", "benchmark", "dm_stat", "p_value_raw", "p_value_holm", "elinear_significantly_better"]]
dm_table.to_csv(f"{RESULTS_DIR}/dm_significance_tests.csv", index=False)
dm_pairwise

## 6. Model complexity (parameter counts) -- PER HORIZON

Parameter count is **not constant across horizons** for most of these architectures: the output/forecasting head scales with `pred_len` (e.g., ELinear's per-learner Linear layer has `pred_len` outputs). Reporting only h = 1 d's count (as an earlier version of this notebook did) would misrepresent model size at longer horizons -- verified directly: ELinear scales from 87 parameters at h = 1 d to 2,610 at h = 30 d, a 30x difference. Counts are therefore collected separately for every horizon.

In [ ]:
param_counts_by_horizon = {}
for model_key in DL_MODEL_KEYS:
    param_counts_by_horizon[model_key] = {}
    for h in HORIZONS:
        try:
            if model_key == "ELinear":
                model, seq_len = load_elinear(h, SEEDS[0])
                n_params = sum(p.numel() for p in model.parameters())
            else:
                nf = load_nf(model_key, h, SEEDS[0])
                n_params = nf_wrappers.nf_param_count(nf)
            param_counts_by_horizon[model_key][h] = n_params
        except FileNotFoundError:
            print(f"No checkpoint found for {model_key} at h={h} -- skipping")
            continue

with open(f"{RESULTS_DIR}/param_counts_by_horizon.json", "w") as f:
    json.dump(param_counts_by_horizon, f, indent=2)

for model_key, by_h in param_counts_by_horizon.items():
    print(model_key, by_h)

## Notes for reviewers / reproducibility
- Every model -- proposed and benchmark alike -- is fit ONCE on the
  chronologically earlier tuning pool and evaluated via the SAME
  `evaluate_walk_forward` function across the held-out test block. No model
  is refit or updated using test-period data at any point.
- DM test direction: a **negative** DM statistic (with `power=2`) combined
  with `elinear_significantly_better=True` means ELinear's squared errors are
  significantly lower than the competitor's, after Holm-Bonferroni correction.
- Parameter counts (`param_counts.json`) are read directly from the trained
  model objects, not hand-estimated -- feeds Notebook 05's complexity-vs-accuracy
  figure without manual entry.